# MOOC Forum Topic Analysis - BERTopic Replication
Replication of: Efficient topic identification for urgent MOOC Forum posts using BERTopic


## Step 1: Data Loading and Validation


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load dataset
data_path = 'dataset/stanfordMOOCForumPostsSet.xlsx'
df = pd.read_excel(data_path)

print(f"Total posts: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Rename columns for convenience
df['urgency'] = df['Urgency(1-7)']
df['course_type'] = df['CourseType']
df['post_text'] = df['Text']

print(f"\nFirst few rows:")
df.head()


Total posts: 29604
Columns: ['Text', 'Opinion(1/0)', 'Question(1/0)', 'Answer(1/0)', 'Sentiment(1-7)', 'Confusion(1-7)', 'Urgency(1-7)', 'CourseType', 'forum_post_id', 'course_display_name', 'forum_uid', 'created_at', 'post_type', 'anonymous', 'anonymous_to_peers', 'up_count', 'comment_thread_id', 'reads']

First few rows:


,Text,Opinion(1/0),Question(1/0),Answer(1/0),Sentiment(1-7),Confusion(1-7),Urgency(1-7),CourseType,forum_post_id,course_display_name,...,created_at,post_type,anonymous,anonymous_to_peers,up_count,comment_thread_id,reads,urgency,course_type,post_text
0,Interesting! How often we say those things to ...,1,0,0,6.5,2.0,1.5,Education,5225177f2c501f0a00000015,Education/EDUC115N/How_to_Learn_Math,...,2013-09-02 22:55:59,Comment,0.0,0.0,0.0,5221a8262cfae31200000001,41.0,1.5,Education,Interesting! How often we say those things to ...
1,"What is \Algebra as a Math Game\"" or are you j...",0,1,0,4.0,5.0,3.5,Education,5207d0e9935dfc0e0000005e,Education/EDUC115N/How_to_Learn_Math,...,2013-08-11 17:59:05,Comment,0.0,0.0,0.0,520663839df35b0a00000043,55.0,3.5,Education,"What is \Algebra as a Math Game\"" or are you j..."
2,I like the idea of my kids principal who says ...,1,0,0,5.5,3.0,2.5,Education,52052c82d01fec0a00000071,Education/EDUC115N/How_to_Learn_Math,...,2013-08-09 17:53:06,Comment,0.0,0.0,0.0,51e59415e339d716000001a6,25.0,2.5,Education,I like the idea of my kids principal who says ...
3,"From their responses, it seems the students re...",1,0,0,6.0,3.0,2.5,Education,5240a45e067ebf1200000008,Education/EDUC115N/How_to_Learn_Math,...,2013-09-23 20:28:14,CommentThread,0.0,0.0,0.0,NaN,0.0,2.5,Education,"From their responses, it seems the students re..."
4,"The boys loved math, because \there is freedom...",1,0,0,7.0,2.0,3.0,Education,5212c5e2dd10251500000062,Education/EDUC115N/How_to_Learn_Math,...,2013-08-20 01:26:58,CommentThread,0.0,0.0,0.0,NaN,3.0,3.0,Education,"The boys loved math, because \there is freedom..."


In [3]:
# Data validation: check urgency distribution
print("Urgency distribution:")
print(df['urgency'].value_counts().sort_index())
print(f"\nTotal posts: {len(df)}")
print(f"Missing values:\n{df.isnull().sum()}")

# Remove rows with missing text
df = df[df['post_text'].notna()].copy()
print(f"\nAfter removing missing text: {len(df)} posts")


Urgency distribution:
urgency
1.0    3501
1.5    3946
2.0    6427
2.5    4624
3.0    3308
3.5    1380
4.0     812
4.5     862
5.0    2259
5.5    1990
6.0     415
6.5      66
7.0      14
Name: count, dtype: int64

Total posts: 29604
Missing values:
Text                       1
Opinion(1/0)               0
Question(1/0)              0
Answer(1/0)                0
Sentiment(1-7)             0
Confusion(1-7)             0
Urgency(1-7)               0
CourseType                 2
forum_post_id             14
course_display_name       13
forum_uid                 19
created_at                19
post_type                 19
anonymous                 19
anonymous_to_peers        19
up_count                  19
comment_thread_id      12306
reads                     19
urgency                    0
course_type                2
post_text                  1
dtype: int64

After removing missing text: 29603 posts


## Step 2: Define Analysis Groups


In [4]:
# Filter urgent posts (urgency >= 4)
df_urgent = df[df['urgency'] >= 4].copy()
print(f"Urgent posts (urgency >= 4): {len(df_urgent)}")
print(f"Expected ~6,415 posts")

# Check course types
print(f"\nCourse types available:")
print(df_urgent['course_type'].value_counts())


Urgent posts (urgency >= 4): 6418
Expected ~6,415 posts

Course types available:
course_type
Medicine      3480
Humanities    2476
Education      461
Name: count, dtype: int64


In [5]:
# Create four analysis groups
groups = {
    'All': df_urgent,
    'Education': df_urgent[df_urgent['course_type'] == 'Education'],
    'Humanities': df_urgent[df_urgent['course_type'] == 'Humanities'],
    'Medicine': df_urgent[df_urgent['course_type'] == 'Medicine']
}

for name, group_df in groups.items():
    print(f"{name}: {len(group_df)} posts")


All: 6418 posts
Education: 461 posts
Humanities: 2476 posts
Medicine: 3480 posts


## Step 3: Data Preprocessing


In [6]:
import re
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import spacy

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load spaCy model for POS tagging
try:
    nlp = spacy.load('en_core_web_sm')
except:
    print("Downloading spaCy model...")
    import os
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

stop_words = set(stopwords.words('english'))


In [7]:
def preprocess_text(text):
    """
    Preprocessing pipeline as per paper:
    expand → clean → stopwords → lowercase → POS → tokenize
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Step 1: Expand contractions (can't -> cannot)
    text = contractions.fix(text)
    
    # Step 2: Remove links
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Step 3: Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Step 4: Remove emojis and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Step 5: Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 6: Lowercase
    text = text.lower()
    
    # Step 7: POS tagging and keep only nouns, adjectives, verbs, adverbs
    doc = nlp(text)
    allowed_pos = {'NOUN', 'ADJ', 'VERB', 'ADV'}
    tokens = [token.text for token in doc if token.pos_ in allowed_pos]
    
    # Step 8: Remove stopwords
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    return ' '.join(tokens)

# Test preprocessing
sample_text = "I can't understand this lecture! Please help ASAP. Check https://example.com"
print(f"Original: {sample_text}")
print(f"Processed: {preprocess_text(sample_text)}")


Original: I can't understand this lecture! Please help ASAP. Check https://example.com
Processed: understand lecture help soon possible check


In [8]:
from tqdm import tqdm
tqdm.pandas()

# Apply preprocessing to all groups
for name, group_df in groups.items():
    print(f"\nProcessing {name} group...")
    groups[name] = group_df.copy()
    groups[name]['cleaned_text'] = groups[name]['post_text'].progress_apply(preprocess_text)
    
    # Remove empty texts
    groups[name] = groups[name][groups[name]['cleaned_text'].str.len() > 0].copy()
    print(f"{name} after cleaning: {len(groups[name])} posts")



Processing All group...


  0%|          | 0/6418 [00:00<?, ?it/s]

100%|██████████| 6418/6418 [01:16<00:00, 84.40it/s] 


All after cleaning: 6416 posts

Processing Education group...


100%|██████████| 461/461 [00:05<00:00, 85.95it/s]


Education after cleaning: 461 posts

Processing Humanities group...


100%|██████████| 2476/2476 [00:40<00:00, 60.91it/s] 


Humanities after cleaning: 2476 posts

Processing Medicine group...


100%|██████████| 3480/3480 [00:43<00:00, 80.15it/s] 


Medicine after cleaning: 3478 posts


## Step 4: Sentence Transformer Embedding


In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model (all-MiniLM-L6-v2)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

  2025-10-24T02:37:56.362458Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7fb0ccf2cea0>), traceback: Some(<traceback object at 0x7fb004e1dc80>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28



In [ ]:
# Generate embeddings for all groups
embeddings = {}
for name, group_df in groups.items():
    print(f"\nGenerating embeddings for {name}...")
    texts = group_df['cleaned_text'].tolist()
    embeddings[name] = embedding_model.encode(texts, show_progress_bar=True)
    print(f"{name} embeddings shape: {embeddings[name].shape}")


## Step 5: Dimensionality Reduction (UMAP)


In [ ]:
from umap import UMAP

# Configure UMAP as per paper
def create_umap_model(n_samples):
    n_neighbors = min(15, n_samples - 1)
    return UMAP(
        n_components=5,
        metric='cosine',
        min_dist=0.05,
        random_state=100,
        n_neighbors=n_neighbors
    )

print("UMAP configuration ready")


## Step 6 & 7: BERTopic Configuration and Training


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN

def create_bertopic_model(n_samples):
    vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
    umap_model = create_umap_model(n_samples)
    hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', prediction_data=True)
    
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=True,
        nr_topics='auto',
        verbose=True
    )
    
    return topic_model

print("BERTopic configuration ready")


In [ ]:
# Train BERTopic for all groups
topic_models = {}
topics_dict = {}
probs_dict = {}

for name, group_df in groups.items():
    print(f"\n{'='*60}")
    print(f"Training BERTopic for {name} group ({len(group_df)} posts)...")
    print(f"{'='*60}")
    
    texts = group_df['cleaned_text'].tolist()
    embs = embeddings[name]
    
    # Create and fit model
    topic_model = create_bertopic_model(len(texts))
    topics, probs = topic_model.fit_transform(texts, embs)
    
    # Store results
    topic_models[name] = topic_model
    topics_dict[name] = topics
    probs_dict[name] = probs
    
    # Basic info
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    print(f"\n{name} Results:")
    print(f"  Number of topics: {n_topics}")
    print(f"  Outliers: {sum(1 for t in topics if t == -1)}")
    print(f"\nTop topics:")
    print(topic_model.get_topic_info().head(10))


## Step 8: Evaluation Metrics


In [ ]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

# Custom RBO implementation (rbo package incompatible with numpy 2.x)
def rbo_score(list1, list2, p=0.9):
    """
    Calculate Rank-Biased Overlap (RBO) between two ranked lists
    p: weighting parameter (0 < p < 1), typically 0.9
    """
    if not list1 or not list2:
        return 0.0
    
    k = min(len(list1), len(list2))
    if k == 0:
        return 0.0
    
    # Calculate agreement at each depth
    overlap = 0.0
    for d in range(1, k + 1):
        set1 = set(list1[:d])
        set2 = set(list2[:d])
        overlap += (len(set1 & set2) / d) * (p ** (d - 1))
    
    # Normalize
    rbo = ((1 - p) / p) * overlap
    return min(1.0, rbo)

print("Custom RBO implementation ready")


In [ ]:
def calculate_coherence_cv(texts, topics_list, topic_model, top_n=10):
    """Calculate C_v coherence score for BERTopic model"""
    tokenized_texts = [text.split() for text in texts]
    dictionary = Dictionary(tokenized_texts)
    
    # Get topics (excluding -1 outlier topic)
    topic_words = []
    unique_topics = sorted(set(topics_list))
    if -1 in unique_topics:
        unique_topics.remove(-1)
    
    for topic_id in unique_topics:
        words = topic_model.get_topic(topic_id)
        if words:
            topic_words.append([word for word, _ in words[:top_n]])
    
    if not topic_words:
        return 0.0
    
    # Calculate coherence
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    
    return coherence_model.get_coherence()

def calculate_irbo(topic_model, top_n=10, p=0.9):
    """Calculate IRBO (Inverted RBO) for topic diversity"""
    topic_info = topic_model.get_topic_info()
    topic_ids = topic_info[topic_info['Topic'] != -1]['Topic'].tolist()
    
    if len(topic_ids) < 2:
        return 1.0
    
    # Get top N words for each topic
    topic_words_lists = []
    for topic_id in topic_ids:
        words = topic_model.get_topic(topic_id)
        if words:
            topic_words_lists.append([word for word, _ in words[:top_n]])
    
    # Calculate pairwise RBO similarities
    similarities = []
    for list1, list2 in combinations(topic_words_lists, 2):
        sim = rbo_score(list1, list2, p=p)
        similarities.append(sim)
    
    if not similarities:
        return 1.0
    
    # IRBO = 1 - mean similarity (higher means more diverse)
    mean_similarity = np.mean(similarities)
    irbo = 1 - mean_similarity
    
    return irbo

print("Evaluation functions ready")


In [ ]:
# Calculate metrics for all groups
results = {}
TOP_N = 10  # Top N words for evaluation

for name in groups.keys():
    print(f"\nEvaluating {name}...")
    
    texts = groups[name]['cleaned_text'].tolist()
    topics_list = topics_dict[name]
    topic_model = topic_models[name]
    
    # Calculate metrics
    coherence = calculate_coherence_cv(texts, topics_list, topic_model, top_n=TOP_N)
    irbo_score_val = calculate_irbo(topic_model, top_n=TOP_N)
    
    n_topics = len(set(topics_list)) - (1 if -1 in topics_list else 0)
    
    results[name] = {
        'n_topics': n_topics,
        'coherence_cv': coherence,
        'irbo': irbo_score_val,
        'n_posts': len(texts)
    }
    
    print(f"  Topics: {n_topics}")
    print(f"  Coherence (C_v): {coherence:.4f}")
    print(f"  IRBO: {irbo_score_val:.4f}")


## Step 9: Results Comparison with Paper


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create results summary table
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(results_df)
print("\n" + "="*80)
print("PAPER REFERENCE VALUES (All urgent posts):")
print("="*80)
print("Topics: ~50-57")
print("Coherence (C_v): ~0.616")
print("IRBO: ~1.0")
print("="*80)

# Save results
results_df.to_csv('results_summary.csv')
print("\nResults saved to: results_summary.csv")


In [ ]:
# Visualize results comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Number of topics
axes[0].bar(results_df.index, results_df['n_topics'], color='skyblue')
axes[0].set_title('Number of Topics by Group')
axes[0].set_ylabel('Number of Topics')
axes[0].tick_params(axis='x', rotation=45)

# Coherence
axes[1].bar(results_df.index, results_df['coherence_cv'], color='lightcoral')
axes[1].axhline(y=0.616, color='red', linestyle='--', label='Paper (All): 0.616')
axes[1].set_title('Topic Coherence (C_v)')
axes[1].set_ylabel('Coherence Score')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

# IRBO
axes[2].bar(results_df.index, results_df['irbo'], color='lightgreen')
axes[2].axhline(y=1.0, color='green', linestyle='--', label='Paper (All): 1.0')
axes[2].set_title('Topic Diversity (IRBO)')
axes[2].set_ylabel('IRBO Score')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend()

plt.tight_layout()
plt.savefig('results_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved to: results_comparison.png")


In [ ]:
# Display top topics for each group
for name in groups.keys():
    print(f"\n{'='*80}")
    print(f"TOP 10 TOPICS - {name.upper()} GROUP")
    print(f"{'='*80}")
    topic_info = topic_models[name].get_topic_info()
    
    # Exclude outlier topic (-1)
    topic_info_filtered = topic_info[topic_info['Topic'] != -1].head(10)
    
    for idx, row in topic_info_filtered.iterrows():
        topic_id = row['Topic']
        count = row['Count']
        words = topic_models[name].get_topic(topic_id)
        top_words = [word for word, _ in words[:10]]
        print(f"\nTopic {topic_id} ({count} posts):")
        print(f"  Keywords: {', '.join(top_words)}")


In [ ]:
# Visualize topics for "All" group (main result)
try:
    fig = topic_models['All'].visualize_topics()
    fig.write_html('topic_visualization_all.html')
    print("Topic visualization saved to: topic_visualization_all.html")
except Exception as e:
    print(f"Could not generate topic visualization: {e}")


In [ ]:
# Save models for future use
import os

os.makedirs('models', exist_ok=True)

for name, model in topic_models.items():
    model.save(f'models/bertopic_{name.lower()}')
    print(f"Model saved: models/bertopic_{name.lower()}")

# Save processed data
for name, group_df in groups.items():
    group_df.to_csv(f'models/processed_data_{name.lower()}.csv', index=False)
    print(f"Data saved: models/processed_data_{name.lower()}.csv")

print("\nAll models and data saved successfully!")


In [ ]:
# Data validation: check urgency distribution
print("Urgency distribution:")
print(df['Urgency(1-7)'].value_counts().sort_index())
print(f"\nTotal posts: {len(df)}")
print(f"Missing values:\n{df.isnull().sum()}")
df['urgency'] = df['Urgency(1-7)']
df['course_type'] = df['CourseType']
df.drop(columns=['Urgency(1-7)', 'CourseType'], inplace=True)
df.head()

Urgency distribution:
Urgency(1-7)
1.0    3501
1.5    3946
2.0    6427
2.5    4624
3.0    3308
3.5    1380
4.0     812
4.5     862
5.0    2259
5.5    1990
6.0     415
6.5      66
7.0      14
Name: count, dtype: int64

Total posts: 29604
Missing values:
Text                       1
Opinion(1/0)               0
Question(1/0)              0
Answer(1/0)                0
Sentiment(1-7)             0
Confusion(1-7)             0
Urgency(1-7)               0
CourseType                 2
forum_post_id             14
course_display_name       13
forum_uid                 19
created_at                19
post_type                 19
anonymous                 19
anonymous_to_peers        19
up_count                  19
comment_thread_id      12306
reads                     19
dtype: int64


,Text,Opinion(1/0),Question(1/0),Answer(1/0),Sentiment(1-7),Confusion(1-7),forum_post_id,course_display_name,forum_uid,created_at,post_type,anonymous,anonymous_to_peers,up_count,comment_thread_id,reads,urgency,course_type
0,Interesting! How often we say those things to ...,1,0,0,6.5,2.0,5225177f2c501f0a00000015,Education/EDUC115N/How_to_Learn_Math,30CADB93E6DE4711193D7BD05F2AE95C,2013-09-02 22:55:59,Comment,0.0,0.0,0.0,5221a8262cfae31200000001,41.0,1.5,Education
1,"What is \Algebra as a Math Game\"" or are you j...",0,1,0,4.0,5.0,5207d0e9935dfc0e0000005e,Education/EDUC115N/How_to_Learn_Math,37D8FAEE7D0B94B6CFC57D98FD3D0BA5,2013-08-11 17:59:05,Comment,0.0,0.0,0.0,520663839df35b0a00000043,55.0,3.5,Education
2,I like the idea of my kids principal who says ...,1,0,0,5.5,3.0,52052c82d01fec0a00000071,Education/EDUC115N/How_to_Learn_Math,CC11480215042B3EB6E5905EAB13B733,2013-08-09 17:53:06,Comment,0.0,0.0,0.0,51e59415e339d716000001a6,25.0,2.5,Education
3,"From their responses, it seems the students re...",1,0,0,6.0,3.0,5240a45e067ebf1200000008,Education/EDUC115N/How_to_Learn_Math,C717F838D10E8256D7C88B33C43623F1,2013-09-23 20:28:14,CommentThread,0.0,0.0,0.0,NaN,0.0,2.5,Education
4,"The boys loved math, because \there is freedom...",1,0,0,7.0,2.0,5212c5e2dd10251500000062,Education/EDUC115N/How_to_Learn_Math,F83887D68EA48964687C6441782CDD0E,2013-08-20 01:26:58,CommentThread,0.0,0.0,0.0,NaN,3.0,3.0,Education


## Step 2: Define Analysis Groups


In [ ]:
# Filter urgent posts (urgency > 4)
df_urgent = df[df['urgency'] >= 4].copy()
print(f"Urgent posts (urgency >= 4): {len(df_urgent)}")
print(f"Expected ~6,415 posts")

# Check course types
print(f"\nCourse types available:")
print(df_urgent['course_type'].value_counts())

Urgent posts (urgency >= 4): 6418
Expected ~6,415 posts

Course types available:
course_type
Medicine      3480
Humanities    2476
Education      461
Name: count, dtype: int64


## Step 3: Data Preprocessing


In [ ]:
import re
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import spacy

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# Load spaCy model for POS tagging
try:
    nlp = spacy.load('en_core_web_sm')
except:
    print("Downloading spaCy model...")
    import os
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

stop_words = set(stopwords.words('english'))


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
def preprocess_text(text):
    """
    Preprocessing pipeline as per paper:
    expand → clean → stopwords → lowercase → POS → tokenize
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Step 1: Expand contractions (can't -> cannot)
    text = contractions.fix(text)
    
    # Step 2: Remove links
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Step 3: Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Step 4: Remove emojis and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Step 5: Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 6: Lowercase
    text = text.lower()
    
    # Step 7: POS tagging and keep only nouns, adjectives, verbs, adverbs
    doc = nlp(text)
    allowed_pos = {'NOUN', 'ADJ', 'VERB', 'ADV'}
    tokens = [token.text for token in doc if token.pos_ in allowed_pos]
    
    # Step 8: Remove stopwords
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    return ' '.join(tokens)

# Test preprocessing
sample_text = "I can't understand this lecture! Please help ASAP. Check https://example.com"
print(f"Original: {sample_text}")
print(f"Processed: {preprocess_text(sample_text)}")


In [ ]:
from tqdm import tqdm
tqdm.pandas()

# Apply preprocessing to all groups
for name, group_df in groups.items():
    print(f"\nProcessing {name} group...")
    groups[name]['cleaned_text'] = group_df['post_text'].progress_apply(preprocess_text)
    
    # Remove empty texts
    groups[name] = groups[name][groups[name]['cleaned_text'].str.len() > 0].copy()
    print(f"{name} after cleaning: {len(groups[name])} posts")


## Step 4: Sentence Transformer Embedding


In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model (all-MiniLM-L6-v2)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")


In [ ]:
# Generate embeddings for all groups
embeddings = {}
for name, group_df in groups.items():
    print(f"\nGenerating embeddings for {name}...")
    texts = group_df['cleaned_text'].tolist()
    embeddings[name] = embedding_model.encode(texts, show_progress_bar=True)
    print(f"{name} embeddings shape: {embeddings[name].shape}")


## Step 5: Dimensionality Reduction (UMAP)


In [ ]:
from umap import UMAP

# Configure UMAP as per paper: cosine metric, min_dist=0.05, n_components=5, random_state=100
# Note: n_neighbors=0.7 doesn't make sense (should be int), using default=15

def create_umap_model(n_samples):
    # Calculate n_neighbors as a reasonable value (not exceeding n_samples)
    n_neighbors = min(15, n_samples - 1)
    return UMAP(
        n_components=5,
        metric='cosine',
        min_dist=0.05,
        random_state=100,
        n_neighbors=n_neighbors
    )

print("UMAP configuration ready")


## Step 6 & 7: BERTopic Configuration and Training


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN

# Configure BERTopic components
def create_bertopic_model(n_samples):
    # Vectorizer with ngram_range=(1,3)
    vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words='english')
    
    # UMAP model
    umap_model = create_umap_model(n_samples)
    
    # HDBSCAN for clustering
    hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', prediction_data=True)
    
    # Create BERTopic model
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=True,
        nr_topics='auto',
        verbose=True
    )
    
    return topic_model

print("BERTopic configuration ready")


In [ ]:
# Train BERTopic for all groups
topic_models = {}
topics_dict = {}
probs_dict = {}

for name, group_df in groups.items():
    print(f"\n{'='*60}")
    print(f"Training BERTopic for {name} group ({len(group_df)} posts)...")
    print(f"{'='*60}")
    
    texts = group_df['cleaned_text'].tolist()
    embs = embeddings[name]
    
    # Create and fit model
    topic_model = create_bertopic_model(len(texts))
    topics, probs = topic_model.fit_transform(texts, embs)
    
    # Store results
    topic_models[name] = topic_model
    topics_dict[name] = topics
    probs_dict[name] = probs
    
    # Basic info
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    print(f"\n{name} Results:")
    print(f"  Number of topics: {n_topics}")
    print(f"  Outliers: {sum(1 for t in topics if t == -1)}")
    print(f"\nTop topics:")
    print(topic_model.get_topic_info().head(10))


## Step 8: Evaluation Metrics


In [ ]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
import rbo
from itertools import combinations

def calculate_coherence_cv(texts, topics_list, topic_model, top_n=10):
    """
    Calculate C_v coherence score for BERTopic model
    """
    # Tokenize texts
    tokenized_texts = [text.split() for text in texts]
    
    # Create dictionary
    dictionary = Dictionary(tokenized_texts)
    
    # Get topics (excluding -1 outlier topic)
    topic_words = []
    unique_topics = sorted(set(topics_list))
    if -1 in unique_topics:
        unique_topics.remove(-1)
    
    for topic_id in unique_topics:
        words = topic_model.get_topic(topic_id)
        if words:
            topic_words.append([word for word, _ in words[:top_n]])
    
    if not topic_words:
        return 0.0
    
    # Calculate coherence
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    
    return coherence_model.get_coherence()

def calculate_irbo(topic_model, top_n=10, p=0.9):
    """
    Calculate IRBO (Inverted Rank-Biased Overlap) for topic diversity
    IRBO = 1 - mean(RBO similarities) where higher means more diverse
    """
    # Get all topics
    topic_info = topic_model.get_topic_info()
    topic_ids = topic_info[topic_info['Topic'] != -1]['Topic'].tolist()
    
    if len(topic_ids) < 2:
        return 1.0
    
    # Get top N words for each topic
    topic_words_lists = []
    for topic_id in topic_ids:
        words = topic_model.get_topic(topic_id)
        if words:
            topic_words_lists.append([word for word, _ in words[:top_n]])
    
    # Calculate pairwise RBO similarities
    similarities = []
    for list1, list2 in combinations(topic_words_lists, 2):
        try:
            sim = rbo.RankingSimilarity(list1, list2).rbo(p=p)
            similarities.append(sim)
        except:
            continue
    
    if not similarities:
        return 1.0
    
    # IRBO = 1 - mean similarity (higher means more diverse)
    mean_similarity = np.mean(similarities)
    irbo = 1 - mean_similarity
    
    return irbo

print("Evaluation functions ready")


In [ ]:
# Calculate metrics for all groups
results = {}
TOP_N = 10  # Top N words for evaluation

for name in groups.keys():
    print(f"\nEvaluating {name}...")
    
    texts = groups[name]['cleaned_text'].tolist()
    topics_list = topics_dict[name]
    topic_model = topic_models[name]
    
    # Calculate metrics
    coherence = calculate_coherence_cv(texts, topics_list, topic_model, top_n=TOP_N)
    irbo_score = calculate_irbo(topic_model, top_n=TOP_N)
    
    n_topics = len(set(topics_list)) - (1 if -1 in topics_list else 0)
    
    results[name] = {
        'n_topics': n_topics,
        'coherence_cv': coherence,
        'irbo': irbo_score,
        'n_posts': len(texts)
    }
    
    print(f"  Topics: {n_topics}")
    print(f"  Coherence (C_v): {coherence:.4f}")
    print(f"  IRBO: {irbo_score:.4f}")


## Step 9: Results Comparison with Paper


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create results summary table
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(results_df)
print("\n" + "="*80)
print("PAPER REFERENCE VALUES (All urgent posts):")
print("="*80)
print("Topics: ~50-57")
print("Coherence (C_v): ~0.616")
print("IRBO: ~1.0")
print("="*80)

# Save results
results_df.to_csv('results_summary.csv')
print("\nResults saved to: results_summary.csv")


In [ ]:
# Visualize results comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Number of topics
axes[0].bar(results_df.index, results_df['n_topics'], color='skyblue')
axes[0].set_title('Number of Topics by Group')
axes[0].set_ylabel('Number of Topics')
axes[0].tick_params(axis='x', rotation=45)

# Coherence
axes[1].bar(results_df.index, results_df['coherence_cv'], color='lightcoral')
axes[1].axhline(y=0.616, color='red', linestyle='--', label='Paper (All): 0.616')
axes[1].set_title('Topic Coherence (C_v)')
axes[1].set_ylabel('Coherence Score')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

# IRBO
axes[2].bar(results_df.index, results_df['irbo'], color='lightgreen')
axes[2].axhline(y=1.0, color='green', linestyle='--', label='Paper (All): 1.0')
axes[2].set_title('Topic Diversity (IRBO)')
axes[2].set_ylabel('IRBO Score')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend()

plt.tight_layout()
plt.savefig('results_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved to: results_comparison.png")


In [ ]:
# Display top topics for each group
for name in groups.keys():
    print(f"\n{'='*80}")
    print(f"TOP 10 TOPICS - {name.upper()} GROUP")
    print(f"{'='*80}")
    topic_info = topic_models[name].get_topic_info()
    
    # Exclude outlier topic (-1)
    topic_info_filtered = topic_info[topic_info['Topic'] != -1].head(10)
    
    for idx, row in topic_info_filtered.iterrows():
        topic_id = row['Topic']
        count = row['Count']
        words = topic_models[name].get_topic(topic_id)
        top_words = [word for word, _ in words[:10]]
        print(f"\nTopic {topic_id} ({count} posts):")
        print(f"  Keywords: {', '.join(top_words)}")


In [ ]:
# Visualize topics for "All" group (main result)
try:
    fig = topic_models['All'].visualize_topics()
    fig.write_html('topic_visualization_all.html')
    print("Topic visualization saved to: topic_visualization_all.html")
except Exception as e:
    print(f"Could not generate topic visualization: {e}")


In [ ]:
# Save models for future use
import pickle
import os

os.makedirs('models', exist_ok=True)

for name, model in topic_models.items():
    model.save(f'models/bertopic_{name.lower()}')
    print(f"Model saved: models/bertopic_{name.lower()}")

# Save processed data
for name, group_df in groups.items():
    group_df.to_csv(f'models/processed_data_{name.lower()}.csv', index=False)
    print(f"Data saved: models/processed_data_{name.lower()}.csv")

print("\nAll models and data saved successfully!")
